In [2]:
import numpy as np 
import pandas as pd 
df=pd.read_csv(r"C:\Users\AARISH ANSARI\Downloads\data_file\marketing_campaign_data_messy.csv")
df.columns

Index([' Campaign_ID ', 'Campaign_Name', 'Start_Date', 'End_Date', 'Channel',
       'Impressions', 'Clicks ', 'Spend', 'Conversions', 'Active', 'Clicks',
       'Campaign_Tag'],
      dtype='str')

In [3]:
df.columns=df.columns.str.strip().str.capitalize().str.replace(" ",'_')

In [4]:
df.columns

Index(['Campaign_id', 'Campaign_name', 'Start_date', 'End_date', 'Channel',
       'Impressions', 'Clicks', 'Spend', 'Conversions', 'Active', 'Clicks',
       'Campaign_tag'],
      dtype='str')

In [2]:
df['Spend']=df['Spend'].str.replace("$","")

In [4]:
df['Spend'].dtype

<StringDtype(storage='python', na_value=nan)>

In [5]:
df.head(3)

,Campaign_ID,Campaign_Name,Start_Date,End_Date,Channel,Impressions,Clicks,Spend,Conversions,Active,Clicks,Campaign_Tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24 00:00:00,2023-12-13,TikTok,16795,197,102.82,20.0,Y,NaN,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06 00:00:00,2023-05-12,Facebook,1860,30,24.33,1.0,0,NaN,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13 00:00:00,2023-12-20,Email,77820,843,1323.39,51.0,No,NaN,EM


In [13]:
df['Channel'].unique()
cleaning_data={
    'Tik_Tok':'TikTok',
    'Gogle':'Google Ads',
    'Insta_gram': 'Instagram',
    'E-mail':'Email',
    'Facebok':'Facebook',
    'N/A': np.nan}
df['Channel']=df['Channel'].replace(cleaning_data)

In [15]:
df['Channel'].value_counts()

Channel
TikTok        415
Facebook      405
Email         380
Google Ads    360
Instagram     359
Name: count, dtype: int64

In [21]:
df['Active'].unique()
bool_map={
    'Y':True,
    '0':False,
    'No':False,
    'Yes':True,
    '1':True,
      1:True,
    0:False}
df['Active']=df['Active'].map(bool_map).fillna(False).astype(bool)

In [22]:
df['Active'].unique()

array([ True, False])

In [25]:
df['Start_Date'] = pd.to_datetime(df['Start_Date'], errors='coerce')
df['End_Date'] = pd.to_datetime(df['End_Date'], dayfirst=True, errors='coerce')

In [26]:
df['Start_Date'].dtype

dtype('<M8[us]')

In [30]:
df['End_Date'].dtype
df

,Campaign_ID,Campaign_Name,Start_Date,End_Date,Channel,Impressions,Clicks,Spend,Conversions,Active,Clicks,Campaign_Tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24,2023-12-13,TikTok,16795,197,$102.82,20.0,True,NaN,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06,2023-05-12,Facebook,1860,30,24.33,1.0,False,NaN,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13,2023-12-20,Email,77820,843,1323.39,51.0,False,NaN,EM
3,CMP-00004,Q1_BlackFriday_CMP-00004,NaT,2023-11-03,TikTok,55886,2019,2180.38,135.0,False,NaN,TI
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22,2023-04-23,Facebook,7265,169,252.44,30.0,True,NaN,FA
...,...,...,...,...,...,...,...,...,...,...,...,...
2015,CMP-00400,Q3_Summer_CMP-00400,2023-10-31,2023-11-13,TikTok,30592,586,$503.95,77.0,True,NaN,TI
2016,CMP-01255,Q4_Summer_CMP-01255,2023-09-01,2023-09-26,Google Ads,20097,897,1641.0,162.0,False,NaN,GO
2017,CMP-01050,Q2_Launch_CMP-01050,2023-02-09,2023-02-21,Instagram,33254,1117,883.82,214.0,False,NaN,IN
2018,CMP-01118,Q4_Winter_CMP-01118,2023-03-30,2023-04-27,Facebook,68728,2960,4198.5,591.0,True,NaN,FA


In [42]:
df.loc[:, df.columns == 'Clicks']
df = df.loc[:, ~df.columns.duplicated(keep='first')]
df.head(2)

,Campaign_id,Campaign_name,Start_date,End_date,Channel,Impressions,Clicks,Spend,Conversions,Active,Campaign_tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24,2023-12-13,TikTok,16795,197,$102.82,20.0,True,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06,2023-05-12,Facebook,1860,30,24.33,1.0,False,FA


In [46]:
impossible_mask=df['Clicks']>df['Impressions']
print(df.loc[impossible_mask,['Campaign_id','Impressions','Clicks']].head(3))

Empty DataFrame
Columns: [Campaign_id, Impressions, Clicks]
Index: []


In [49]:
time_mask=df['End_date']<df['Start_date']
print(df.loc[time_mask,['Campaign_id','Start_date','End_date']].head(3))
df.loc[time_mask,'End_date']=df.loc[time_mask,'Start_date']+pd.Timedelta(days=30)
print(df.loc[time_mask,['Campaign_id','Start_date','End_date']].head(3))

Empty DataFrame
Columns: [Campaign_id, Start_date, End_date]
Index: []
Empty DataFrame
Columns: [Campaign_id, Start_date, End_date]
Index: []


In [16]:
q1=df['Spend'].quantile(0.25)
q2=df['Spend'].quantile(0.75)
IQR=q2-q1
upper_limit=q2+(3*q1)
outlier=df['Spend']>upper_limit
print('fix applied')
df.loc[outlier,'Spend']=upper_limit
print(df.loc[outlier,['Campaign_id','Spend']].head(3))


Empty DataFrame
Columns: [Campaign_id, Spend]
Index: []
fix applied
Empty DataFrame
Columns: [Campaign_id, Spend]
Index: []


In [11]:
df['Spend'].dtype
df['Spend'] = pd.to_numeric(df['Spend'], errors='coerce')

dtype('float64')

In [25]:
df['season'] = df['Campaign_name'].str.extract(r'Q\d_([^_]+)_')
print('fix applied')
print(df[['Campaign_name','season']].head(3))

fix applied
         Campaign_name  season
0  Q4_Summer_CMP-00001  Summer
1  Q1_Launch_CMP-00002  Launch
2  Q3_Winter_CMP-00003  Winter


Project_2